# Sample 1 — every mistake, model by model

The annotation for sample 1 lists a fixed set of items. For each model below:
what it got right, and every single thing it got wrong.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import pandas as pd
from IPython.display import display

from dmpbridge.core import paths as P
from dmpbridge.evaluation.evaluate import (
    _confusion_from_match, _match_structured, extract_gold, micro_prf1,
    resolve_old_gt_path,
)

SAMPLE, EXTRACTOR = 1, 'pdfplumber'
pd.set_option('display.max_colwidth', 90)
gold = extract_gold(resolve_old_gt_path(SAMPLE))


def mistakes(model):
    """Every wrong item this model produced, and the score for the document."""
    tag = P.make_tag(model, EXTRACTOR)
    rec, extra = _match_structured(P.structured_path(tag, SAMPLE), gold)
    m = micro_prf1(_confusion_from_match(rec, extra))
    rows = [{'should be': r['gold_label'], 'model said': r['pred_label'],
             'text': r['pred_text']}
            for r in rec if r['pred_label'] and r['pred_label'] != r['gold_label']]
    rows += [{'should be': 'nothing — extra item', 'model said': l, 'text': t}
             for t, l in extra]
    rows += [{'should be': r['gold_label'], 'model said': 'not produced',
              'text': r['gold_text']}
             for r in rec if r['pred_label'] is None]
    return pd.DataFrame(rows), m


def show(model):
    """Print the score, then list the mistakes."""
    df, m = mistakes(model)
    print(f'{len(gold)} items in the annotation')
    print(f'  {m["tp"]:>2} correct')
    print(f'  {len(df):>2} wrong')
    print(f'  f1 = {m["f1"]:.3f}')
    if len(df):
        print()
        display(df)
    else:
        print('\nNo mistakes.')


print('sections below:', ', '.join(['llama3.1:8b', 'gemma4:e4b', 'llama3.3:70b']))


sections below: llama3.1:8b, gemma4:e4b, llama3.3:70b


## llama3.1:8b


In [2]:
show('llama3.1:8b')


28 items in the annotation
  19 correct
  12 wrong
  f1 = 0.644



,should be,model said,text
0,question.text,section.title,"B. Scientific data that will be preserved and shared, and the rationale for doing so:"
1,question.text,section.title,"C. Metadata, other relevant data, and associated documentation:"
2,answer.text,question.text,The following data will be created as a result of this project:
3,question.text,section.title,A. Repository where scientific data and metadata will be archived:
4,question.text,section.title,B. How scientific data will be findable and identifiable:
5,question.text,section.title,C. When and how long the scientific data will be made available:
6,question.text,section.title,"A. Factors affecting subsequent access, distribution, or reuse of scientific data:"
7,question.text,section.title,B. Whether access to scientific data will be controlled:
8,question.text,section.title,"Protections for privacy, rights, and confidentiality of human research participants:"
9,nothing — extra item,answer.text,Objective sedentary behavior metrics – no existing standards


## gemma4:e4b


In [3]:
show('gemma4:e4b')


28 items in the annotation
  28 correct
   0 wrong
  f1 = 1.000

No mistakes.


## llama3.3:70b


In [4]:
show('llama3.3:70b')


28 items in the annotation
  27 correct
   2 wrong
  f1 = 0.947



,should be,model said,text
0,answer.text,section.description,The following data will be created as a result of this project:
1,nothing — extra item,answer.text,Objective sedentary behavior metrics – no existing standards


---

**How to read the `should be` column**

| value | meaning |
|---|---|
| a label name | the annotation has this text as that label; the model called it something else |
| `nothing — extra item` | the model produced an item the annotation does not have |
| paired with `not produced` | the annotation has this item and the model never produced it |
